# 이모지스튜디오 · 무료 클라우드 실행 준비

이 노트북은 본인의 Colab **무료 GPU**에서 웹 편집기와 MuseTalk 작업 서버를 실행하도록 준비한 초안입니다. 유료 서비스로 전환하지 않습니다. 실제 Colab GPU 설치·생성 품질은 아직 검증 전이며 모든 AI 기능이 완성된 것은 아닙니다.

1. 본인 Google 계정으로 Colab에 로그인합니다.
2. **런타임 → 런타임 유형 변경 → GPU**를 선택합니다. 무료 GPU가 배정되지 않으면 중단하고 나중에 다시 시도하세요.
3. **모두 실행**을 누르고 개인 Drive 연결 요청에 동의합니다. 첫 설치에는 시간이 걸립니다.
4. 마지막 셀의 접속 링크를 열어 ‘연결’을 누릅니다. 또는 ZIP의 index.html 연결 칸에 링크를 붙여 넣습니다.

연결 코드가 포함된 마지막 링크는 본인만 사용하세요. 무료 GPU의 지속시간/재배정 시간은 보장되지 않습니다. 할당 제한을 우회하거나 세션을 강제로 유지하는 기능은 없습니다.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('무료 GPU가 없습니다. 런타임 유형을 GPU로 바꾸고 배정 가능 여부를 확인하세요. 유료 모드로 자동 전환하지 않습니다.')
print('할당된 GPU:', torch.cuda.get_device_name(0))
print('GPU 배정을 확인했습니다. 실제 모델 생성 검증은 아직 별도로 필요합니다.')


## 개인 Drive에 프로젝트 보존
완료된 처리 단위와 내보낸 파일을 개인 Drive의 `EmojiStudio-Projects`에 저장합니다. PC와 Colab의 브라우저 폴더 권한은 별개입니다. 새 접속 주소에서는 PC 저장 폴더를 다시 선택해야 할 수 있습니다. 세션이 종료되면 다시 노트북을 실행해야 하며, 종료·재개 전체 흐름은 아직 검증 전입니다.


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/EmojiStudio-Projects')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('프로젝트 보존 위치:', DATA_DIR)


## 검증 가능한 소스 버전 가져오기
공개 GitHub의 고정 커밋을 사용합니다. 다른 사람이 작성한 기존 파일이나 작업 트리를 초기화하지 않습니다.


In [ ]:
import subprocess
from pathlib import Path
APP_COMMIT = '11dd564b04d3e94eaffc7d130edaf8b7115db647'
SOURCE_ROOT = Path('/content/EmojiStudio-source-' + APP_COMMIT[:12])
if not SOURCE_ROOT.exists():
    subprocess.run(['git', 'clone', '--no-checkout', 'https://github.com/jhpork74-bot/3dmovie.git', str(SOURCE_ROOT)], check=True)
    subprocess.run(['git', 'checkout', '--detach', APP_COMMIT], cwd=SOURCE_ROOT, check=True)
else:
    found = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=SOURCE_ROOT, text=True).strip()
    if found != APP_COMMIT:
        raise RuntimeError('기존 소스 폴더가 다른 버전입니다. 파일을 덮어쓰지 않고 중단합니다.')
APP_ROOT = SOURCE_ROOT / '3D프로그램'
if not (APP_ROOT / 'cloud' / 'colab_launch.py').is_file():
    raise RuntimeError('실행 도우미가 소스에 없습니다. 다운로드를 확인하세요.')
print('소스 버전:', APP_COMMIT[:12])


## 설치 및 서버 시작
공식 HTTPS 배포와 제공된 체크섬을 확인합니다. 검증에 실패하면 우회하지 않고 중단합니다. AI 입력·출력과 설치 로그가 본인 Colab 환경에 생성되며, 사용자 Google 계정이나 비밀번호를 프로그램에 입력할 필요는 없습니다.

이 단계에는 여러 GB의 모델/런타임 다운로드가 포함됩니다. 사이트 제한이나 라이브러리 호환성 문제가 발생하면 셀 출력과 로그를 확인해야 합니다. **서버 프로세스가 시작됐다고 AI 품질 검증이 끝난 것은 아닙니다.**


In [ ]:
import sys
import urllib.parse
from IPython.display import display, HTML
import html
sys.path.insert(0, str(APP_ROOT / 'cloud'))
import colab_launch
# Re-running only stops processes created by this notebook, never unrelated services.
if 'SESSION' in globals():
    for process in reversed(SESSION.get('processes', [])):
        if process.poll() is None:
            process.terminate()
SESSION = colab_launch.launch(APP_ROOT, DATA_DIR)
connection_url = SESSION['url'] + '/#code=' + urllib.parse.quote(SESSION['code'], safe='')
display(HTML('<h3>이모지스튜디오 접속</h3><p><a target="_blank" rel="noopener noreferrer" href="' + html.escape(connection_url, quote=True) + '">내 무료 세션의 웹 편집기 열기</a></p>'))
print('index.html 연결 칸에 붙여 넣을 본인 전용 주소:')
print(connection_url)
print('프로젝트/완료 파일 보관:', SESSION['data_dir'])
print('문제 확인용 설치·서버 로그:', SESSION['logs'])
print('포트 시작만 확인한 상태입니다. 실제 정밀 립싱크 생성과 미완료 AI 기능을 구분하세요.')


## 현재 기능 제한
- 현재 개발본에서 사진 자동 대본·원본 음성 자동 인식·감정 표정 렌더링은 미완료입니다. 관련 화면이 있어도 실제 기능이 완성된 것으로 간주하지 마세요.
- MuseTalk 정밀 립싱크의 실제 구동과 사용자 이모지 얼굴 인식·알파 복원 품질도 아직 검증해야 합니다.
- Colab 세션이 종료되면 주소와 연결 코드가 바뀝니다. 다시 실행해 새 주소를 사용하세요. Drive의 완료 파일을 이용한 전체 재개 검사도 남아 있습니다.
- 모델 다운로드나 공식 체크섬 확인 실패를 무시하지 말고, 출력의 실제 오류를 개발자에게 전달하세요. 비밀번호/접속 코드는 공유하지 마세요.
